# Python functools — Interview Preparation

Covers: lru_cache/cache, partial, reduce, wraps, total_ordering, singledispatch, cached_property, and interview Q&A.

## 1. lru_cache and cache

In [ ]:
from functools import lru_cache, cache
import time

# Without cache — exponential time
def fib_slow(n):
    return n if n < 2 else fib_slow(n-1) + fib_slow(n-2)

# With lru_cache — O(n) time
@lru_cache(maxsize=128)
def fib_cached(n):
    return n if n < 2 else fib_cached(n-1) + fib_cached(n-2)

# With cache (Python 3.9+) — unlimited, slightly faster
@cache
def fib_cache(n):
    return n if n < 2 else fib_cache(n-1) + fib_cache(n-2)


start = time.perf_counter()
print('fib_slow(30):', fib_slow(30))
print(f'  time: {time.perf_counter()-start:.4f}s')

start = time.perf_counter()
print('fib_cached(100):', fib_cached(100))
print(f'  time: {time.perf_counter()-start:.6f}s')

# Cache info
print('cache_info:', fib_cached.cache_info())
fib_cached.cache_clear()
print('after clear:', fib_cached.cache_info())

In [ ]:
# Arguments must be hashable
@lru_cache(maxsize=128)
def process(data):  # data must be hashable
    return sum(data)

print('process((1,2,3)):', process((1, 2, 3)))  # tuple — OK

try:
    process([1, 2, 3])  # list — not hashable!
except TypeError as e:
    print('Error:', e)

# Practical: memoized DP
@cache
def coin_change(amount, coins):
    """Minimum coins to make amount"""
    if amount == 0:
        return 0
    if amount < 0 or not coins:
        return float('inf')
    return min(
        1 + coin_change(amount - coins[0], coins),
        coin_change(amount, coins[1:])
    )

print('coin_change(11, (1,5,6,9)):', coin_change(11, (1, 5, 6, 9)))

## 2. partial

In [ ]:
from functools import partial

def power(base, exponent):
    return base ** exponent

# Fix some arguments
square = partial(power, exponent=2)
cube = partial(power, exponent=3)

print('square(5):', square(5))
print('cube(3):', cube(3))

# Fix positional argument
def multiply(x, y):
    return x * y

double = partial(multiply, 2)  # fixes first positional arg
print('double(7):', double(7))

# Inspect partial
print('double.func:', double.func)
print('double.args:', double.args)
print('double.keywords:', double.keywords)

In [ ]:
# Practical: partial for configuration
import os

join_home = partial(os.path.join, os.path.expanduser('~'))
print('join_home:', join_home('documents', 'file.txt'))

# partial vs lambda — partial is more introspectable
double_lambda = lambda x: multiply(2, x)
double_partial = partial(multiply, 2)

print('lambda __name__:', double_lambda.__name__)   # '<lambda>'
print('partial func:', double_partial.func.__name__)  # 'multiply'

# Use partial with sorted
from operator import itemgetter
sort_by_age = partial(sorted, key=itemgetter('age'))
people = [{'name': 'Alice', 'age': 30}, {'name': 'Bob', 'age': 25}]
print('sorted by age:', sort_by_age(people))

## 3. reduce

In [ ]:
from functools import reduce
import operator

nums = [1, 2, 3, 4, 5]

# Basic reductions
print('sum:', reduce(operator.add, nums))       # 15
print('product:', reduce(operator.mul, nums))   # 120
print('max:', reduce(lambda a, b: a if a > b else b, nums))  # 5

# With initializer
print('sum with init=100:', reduce(operator.add, nums, 100))  # 115
print('empty with init:', reduce(operator.add, [], 0))        # 0

# Function composition
def compose(*funcs):
    """Right-to-left composition: compose(f, g)(x) = f(g(x))"""
    return reduce(lambda f, g: lambda x: f(g(x)), funcs)

add1 = lambda x: x + 1
double = lambda x: x * 2
square = lambda x: x ** 2

# square(double(add1(x)))
transform = compose(square, double, add1)
print('compose(square, double, add1)(3):', transform(3))  # square(double(4)) = square(8) = 64

## 4. total_ordering and singledispatch

In [ ]:
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, major, minor, patch):
        self.major = major
        self.minor = minor
        self.patch = patch

    def __eq__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return (self.major, self.minor, self.patch) == (other.major, other.minor, other.patch)

    def __lt__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return (self.major, self.minor, self.patch) < (other.major, other.minor, other.patch)

    def __hash__(self):
        return hash((self.major, self.minor, self.patch))

    def __repr__(self):
        return f'{self.major}.{self.minor}.{self.patch}'


v1 = Version(1, 2, 3)
v2 = Version(2, 0, 0)
v3 = Version(1, 3, 0)

print('v1 < v2:', v1 < v2)   # True
print('v1 > v2:', v1 > v2)   # False — generated by @total_ordering
print('v1 <= v3:', v1 <= v3) # True — generated
print('sorted:', sorted([v2, v1, v3]))

In [ ]:
from functools import singledispatch

@singledispatch
def process(data):
    raise TypeError(f'Unsupported type: {type(data).__name__}')

@process.register(int)
def _(data):
    return f'int: {data * 2}'

@process.register(str)
def _(data):
    return f'str: {data.upper()}'

@process.register(list)
def _(data):
    return f'list of {len(data)} items: {sum(data) if all(isinstance(x, (int, float)) for x in data) else data}'

@process.register(float)
@process.register(complex)
def _(data):
    return f'number: {abs(data):.2f}'


print(process(42))
print(process('hello'))
print(process([1, 2, 3]))
print(process(3.14))

try:
    process({1: 2})
except TypeError as e:
    print('Error:', e)

## 5. cached_property

In [ ]:
from functools import cached_property

class Circle:
    def __init__(self, radius):
        self.radius = radius

    @cached_property
    def area(self):
        import math
        print('Computing area...')
        return math.pi * self.radius ** 2

    @cached_property
    def circumference(self):
        import math
        print('Computing circumference...')
        return 2 * math.pi * self.radius


c = Circle(5)
print('First access:')
print('area:', c.area)           # computes
print('Second access:')
print('area:', c.area)           # cached — no print!

# Stored in instance __dict__
print('In __dict__:', 'area' in c.__dict__)

# Can be invalidated by deleting from __dict__
del c.__dict__['area']
print('After invalidation:')
print('area:', c.area)           # recomputes

## 6. Interview Practice

In [ ]:
# Q: lru_cache vs cache — when to use which?
from functools import lru_cache, cache

# lru_cache with size limit — for memory-constrained scenarios
@lru_cache(maxsize=100)
def expensive_with_limit(n):
    return n ** 2

# cache — unlimited, for pure functions where all results should be kept
@cache
def expensive_unlimited(n):
    return n ** 2

# Demonstrate LRU eviction
@lru_cache(maxsize=3)
def small_cache(n):
    return n

small_cache(1); small_cache(2); small_cache(3)
print('Cache info (full):', small_cache.cache_info())
small_cache(4)  # evicts 1 (least recently used)
print('Cache info (after 4):', small_cache.cache_info())

In [ ]:
# Q: cached_property vs lru_cache on method — memory leak risk
import gc

# lru_cache on method — keeps reference to self, prevents GC!
class WithLRU:
    @lru_cache(maxsize=None)
    def compute(self):
        return id(self)

# cached_property — stores in instance __dict__, no class-level reference
class WithCachedProp:
    @cached_property
    def compute(self):
        return id(self)

# cached_property is the right choice for instance methods
obj = WithCachedProp()
print('compute:', obj.compute)
print('In __dict__:', 'compute' in obj.__dict__)  # True — stored per-instance

print('\nSummary:')
print('- @cache / @lru_cache: for module-level functions, hashable args required')
print('- @cached_property: for instance methods, stored in instance __dict__')